# Pearls AQI Predictor — Exploratory Data Analysis

This notebook analyzes historical AQI and weather data for Karachi to:
- Identify seasonal and diurnal patterns
- Understand correlations between features
- Determine optimal lag windows (ACF/PACF)
- Detect outliers and data quality issues
- Compute persistence baseline RMSE

**Prerequisites:** Hopsworks credentials must be set in `.env` before running this notebook.

```bash
cp .env.example .env  # then fill in HOPSWORKS_API_KEY, AQICN_API_KEY, OPENWEATHER_API_KEY
```

Run from the `notebooks/` directory, or adjust `sys.path` in the first code cell accordingly.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_squared_error

sns.set_style('darkgrid')
pd.set_option('display.max_columns', 50)
print('Libraries loaded')

## 1. Load Data from Hopsworks Feature Store

In [ ]:
from src.feature_pipeline.store_features import fetch_training_data

df = fetch_training_data()
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)
print(f'Shape: {df.shape}')
df.head()
# day_of_week is not stored as a column (only one-hot dow_0..dow_6 are stored)
# Reconstruct from timestamp for EDA visualisations
df['day_of_week'] = pd.to_datetime(df['timestamp']).dt.dayofweek


## 2. Basic Statistics & Data Quality

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum().sort_values(ascending=False).head(20))
print()
print('=== Descriptive Stats ===')
df[['aqi', 'pm25', 'pm10', 'o3', 'no2', 'so2', 'temperature', 'humidity', 'wind_speed']].describe()

## 3. AQI Time Series + STL Decomposition

In [ ]:
fig = px.line(df, x='timestamp', y='aqi',
              title='AQI Over Time — Karachi',
              labels={'aqi': 'AQI', 'timestamp': 'Date'},
              template='plotly_dark')
# Add AQI zone bands
for lo, hi, label, color in [(0,50,'Good','rgba(0,228,0,0.08)'),
                               (51,100,'Moderate','rgba(255,255,0,0.06)'),
                               (101,150,'Unhealthy (Sensitive)','rgba(255,126,0,0.08)'),
                               (151,200,'Unhealthy','rgba(255,0,0,0.08)'),
                               (201,500,'Hazardous','rgba(126,0,35,0.1)')]:
    fig.add_hrect(y0=lo, y1=hi, fillcolor=color, line_width=0, annotation_text=label, annotation_position='right')
fig.show()

In [ ]:
# STL Decomposition
aqi_series = df.set_index('timestamp')['aqi'].dropna()
if len(aqi_series) >= 48:
    stl = STL(aqi_series, period=24, robust=True)
    result = stl.fit()
    fig, axes = plt.subplots(4, 1, figsize=(14, 10))
    axes[0].plot(aqi_series.index, aqi_series.values, color='#4a90d9'); axes[0].set_title('Original AQI')
    axes[1].plot(result.trend.index, result.trend.values, color='#f5c842'); axes[1].set_title('Trend')
    axes[2].plot(result.seasonal.index, result.seasonal.values, color='#a6e3a1'); axes[2].set_title('Seasonal (24h cycle)')
    axes[3].plot(result.resid.index, result.resid.values, color='#f38ba8'); axes[3].set_title('Residual')
    for ax in axes: ax.set_facecolor('#1e1e2e')
    plt.tight_layout()
    plt.show()

## 4. Diurnal & Seasonal Patterns

In [ ]:
# AQI by hour of day
if 'hour' in df.columns:
    hourly = df.groupby('hour')['aqi'].agg(['mean', 'std']).reset_index()
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=hourly['hour'], y=hourly['mean'] + hourly['std'],
                             fill=None, mode='lines', line_color='rgba(74,144,217,0)', showlegend=False))
    fig.add_trace(go.Scatter(x=hourly['hour'], y=hourly['mean'] - hourly['std'],
                             fill='tonexty', mode='lines', line_color='rgba(74,144,217,0)',
                             fillcolor='rgba(74,144,217,0.2)', name='±1 Std'))
    fig.add_trace(go.Scatter(x=hourly['hour'], y=hourly['mean'], mode='lines+markers',
                             name='Mean AQI', line=dict(color='#4a90d9', width=2)))
    fig.update_layout(title='Mean AQI by Hour of Day', template='plotly_dark',
                      xaxis_title='Hour', yaxis_title='AQI')
    fig.show()

In [ ]:
# AQI heatmap: hour × day_of_week
if 'hour' in df.columns and 'day_of_week' in df.columns:
    pivot = df.pivot_table(values='aqi', index='hour', columns='day_of_week', aggfunc='mean')
    pivot.columns = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    fig = px.imshow(pivot, title='Mean AQI: Hour × Day of Week',
                    color_continuous_scale='RdYlGn_r', template='plotly_dark',
                    labels={'color': 'AQI'})
    fig.show()

## 5. Correlation Heatmap

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
# Keep manageable — focus on key features
key_cols = ['aqi', 'pm25', 'pm10', 'o3', 'no2', 'so2', 'co',
            'temperature', 'humidity', 'pressure', 'wind_speed',
            'aqi_lag_1h', 'aqi_lag_6h', 'aqi_lag_24h',
            'rolling_mean_24h', 'aqi_change_rate']
key_cols = [c for c in key_cols if c in df.columns]

corr = df[key_cols].corr()
fig = px.imshow(corr, title='Feature Correlation Matrix',
                color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                template='plotly_dark')
fig.update_layout(width=800, height=700)
fig.show()

## 6. ACF / PACF — Optimal Lag Selection

In [ ]:
aqi_clean = df['aqi'].dropna()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(aqi_clean, lags=48, ax=ax1, title='ACF — AQI (48h lags)')
plot_pacf(aqi_clean, lags=48, ax=ax2, title='PACF — AQI (48h lags)', method='ywmle')
plt.tight_layout()
plt.show()
print('Key lags from ACF: these inform which lag features are most valuable (significant spikes)')

## 7. Outlier Detection (IQR)

In [ ]:
for col in ['aqi', 'pm25', 'pm10', 'temperature']:
    if col not in df.columns:
        continue
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 3*IQR, Q3 + 3*IQR
    n_outliers = ((df[col] < lo) | (df[col] > hi)).sum()
    print(f'{col:20s}: {n_outliers} outliers ({n_outliers/len(df)*100:.1f}%) | range=[{lo:.1f}, {hi:.1f}]')

## 8. Persistence Baseline RMSE

In [ ]:
if 'aqi_24h' in df.columns:
    valid = df[['aqi', 'aqi_24h']].dropna()
    persistence_preds = valid['aqi'].values  # predict current = future
    actual = valid['aqi_24h'].values
    pers_rmse = np.sqrt(mean_squared_error(actual, persistence_preds))
    print(f'Persistence Baseline RMSE (24h): {pers_rmse:.2f}')
    print('Your trained models must beat this score to be useful.')
    print(f'Equivalent to Skill Score = 0. Models with Skill Score > 0 beat persistence.')

## 9. Pollutant Distributions

In [ ]:
pollutants = [c for c in ['pm25', 'pm10', 'o3', 'no2', 'so2', 'co'] if c in df.columns]
fig = make_subplots(rows=2, cols=3, subplot_titles=pollutants)
for i, p in enumerate(pollutants):
    r, c = i // 3 + 1, i % 3 + 1
    fig.add_trace(go.Histogram(x=df[p].dropna(), name=p, nbinsx=50,
                               marker_color='#4a90d9', opacity=0.8), row=r, col=c)
fig.update_layout(title='Pollutant Distributions (µg/m³)',
                  template='plotly_dark', showlegend=False, height=500)
fig.show()
print('Note: PM2.5 and PM10 are right-skewed → log(x+1) transform is applied before training')